# Module 6 — SOLUTION: Experiment Rounds

> **Instructor note:** This is the complete solution. Share after all three rounds are done.

Three rounds, each isolating one variable:
- **Round 1** — chunking strategy (fixed vs. recursive vs. token-aware)
- **Round 2** — retrieval config (top_k + vector vs. hybrid)
- **Round 3** — prompt template (four variants)

Everything is logged to `ExperimentLog` so results survive kernel restarts.

In [ ]:
import sys, json, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

from solutions.src.chunker import build_chunk_records
from solutions.src.embedder import DEFAULT_MODEL
from solutions.src.vector_store import get_collection, index_chunks
from solutions.src.retriever import hybrid_retrieve as _hybrid_retrieve
from solutions.src.pipeline import RAGPipeline
from solutions.src.evaluator import run_evaluation
from solutions.src.experiment_log import ExperimentLog

load_dotenv("../.env")

FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.com/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)

with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

with open("../data/sample_qa.json") as f:
    qa_data = json.load(f)

log = ExperimentLog()
print(f"ExperimentLog: {len(log)} existing entries")
print(f"Corpus: {len(corpus)} documents | Test questions: {len(qa_data['questions'])}")

In [ ]:
# Pre-load the embedding model so it doesn't silently block mid-experiment
# (~250 MB download on first run — this may take a minute)
from solutions.src.embedder import get_model, DEFAULT_MODEL
print(f"Loading embedding model: {DEFAULT_MODEL}")
_model = get_model(DEFAULT_MODEL)
print(f"Model ready — embedding dim: {_model.get_sentence_embedding_dimension()}")

In [ ]:
def run_experiment(
    name: str,
    collection_name: str,
    chunks: list[dict],
    pipeline_kwargs: dict,
    config_meta: dict,
    notes: str = "",
) -> dict:
    """
    Index chunks, run pipeline on all test questions, evaluate, log result.
    Always drops and recreates the collection so embedding-dim changes never cause errors.
    """
    import chromadb as _chromadb
    _client = _chromadb.PersistentClient(path="../chroma_db")
    try:
        _client.delete_collection(collection_name)
    except Exception:
        pass
    collection = _client.get_or_create_collection(
        name=collection_name, metadata={"hnsw:space": "cosine"}
    )
    index_chunks(collection, chunks, model_name=DEFAULT_MODEL, force=True)

    rag = RAGPipeline(collection=collection, llm_client=llm_client, **pipeline_kwargs)

    results = []
    for qa in tqdm(qa_data["questions"], desc=f"[{name}]"):
        r = rag.ask(qa["question"])
        r["ground_truth"] = qa["ground_truth"]
        results.append(r)

    df = run_evaluation(results, llm_client=llm_client, judge_model=FAST_MODEL)
    metric_cols = [c for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"] if c in df.columns]
    scores = {c: round(float(df[c].mean()), 4) for c in metric_cols}

    log.add(name=name, config=config_meta, scores=scores, notes=notes)
    return scores


print("Helper ready.")

---
## Round 1 — Chunking Strategy

**Hypothesis:** `RecursiveCharacterTextSplitter` preserves section boundaries in structured insurance
documents better than a fixed-size splitter, improving context recall.

**Variable:** chunking strategy and size  
**Fixed:** vector retrieval, top_k=5, baseline prompt

| Config | Strategy | Size |
|--------|----------|------|
| Baseline | `recursive` | 800 chars |
| B | `fixed` | 800 chars (no separator hierarchy) |
| C | `recursive` | 400 chars (smaller chunks) |
| D | `tokens` | 100 tokens (model-native) |

In [ ]:
# ── Baseline: RecursiveCharacterTextSplitter 800 chars ─────────────────────
if not any(e["name"] == "Baseline" for e in log._entries):
    run_experiment(
        name="Baseline",
        collection_name="exp_baseline",
        chunks=build_chunk_records(corpus, strategy="recursive", chunk_size=800, overlap=100),
        pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
        config_meta={"strategy": "recursive", "chunk_size": 800, "overlap": 100, "top_k": 5, "retrieval": "vector"},
        notes="Baseline — recursive 800 chars, vector search",
    )
else:
    print("Baseline already logged — skipping.")

In [ ]:
# ── B: CharacterTextSplitter 800 chars (no separator fallback) ─────────────
run_experiment(
    name="Round1-Fixed-800",
    collection_name="exp_r1_fixed_800",
    chunks=build_chunk_records(corpus, strategy="fixed", chunk_size=800, overlap=100),
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
    config_meta={"strategy": "fixed", "chunk_size": 800, "overlap": 100, "top_k": 5, "retrieval": "vector"},
    notes="CharacterTextSplitter — same size as baseline but no separator hierarchy",
)

In [ ]:
# ── C: RecursiveCharacterTextSplitter 400 chars (smaller chunks) ───────────
run_experiment(
    name="Round1-Recursive-400",
    collection_name="exp_r1_recursive_400",
    chunks=build_chunk_records(corpus, strategy="recursive", chunk_size=400, overlap=50),
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
    config_meta={"strategy": "recursive", "chunk_size": 400, "overlap": 50, "top_k": 5, "retrieval": "vector"},
    notes="Smaller recursive chunks — more precise retrieval, less context per chunk",
)

In [ ]:
# Round 1 summary
log.summary()
log.delta_table(baseline_name="Baseline")
log.plot()

# Pick the winner based on context_recall (retrieval quality)
import pandas as pd
df_log = log.to_dataframe()
r1_entries = df_log[df_log["name"].str.startswith(("Baseline", "Round1"))]
best_r1 = r1_entries.loc[r1_entries["context_recall"].idxmax(), "name"]
print(f"\nRound 1 winner (best context_recall): {best_r1}")

# Set config for Round 2 — update if your results differ
BEST_CHUNK_STRATEGY = "recursive"
BEST_CHUNK_KWARGS = {"chunk_size": 800, "overlap": 100}
best_chunks = build_chunk_records(corpus, strategy=BEST_CHUNK_STRATEGY, **BEST_CHUNK_KWARGS)
print(f"Using for Round 2: strategy={BEST_CHUNK_STRATEGY}, {len(best_chunks)} chunks")

---
## Round 2 — Retrieval Configuration

**Hypothesis:** Hybrid search (BM25 + vector) improves precision for technical insurance terms
(INAMI, RIZIV, franchise, vrijstelling) that carry meaning through exact spelling, not just semantics.

**Variable:** retrieval method and top_k  
**Fixed:** best chunking from Round 1, baseline prompt

| Config | Retrieval | top_k |
|--------|-----------|-------|
| E | vector | 3 |
| F | vector | 8 |
| G | hybrid (BM25 + vector, RRF) | 5 |

In [ ]:
# ── E: Vector top_k=3 (high precision, lower recall) ──────────────────────
run_experiment(
    name="Round2-Vector-k3",
    collection_name="exp_r2_vector_k3",
    chunks=best_chunks,
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 3},
    config_meta={"strategy": BEST_CHUNK_STRATEGY, "top_k": 3, "retrieval": "vector"},
    notes="Fewer context chunks — less noise but may miss relevant passages",
)

In [ ]:
# ── F: Vector top_k=8 (lower precision, higher recall) ────────────────────
run_experiment(
    name="Round2-Vector-k8",
    collection_name="exp_r2_vector_k8",
    chunks=best_chunks,
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 8},
    config_meta={"strategy": BEST_CHUNK_STRATEGY, "top_k": 8, "retrieval": "vector"},
    notes="More context — better recall but LLM has more material to sift through",
)

In [ ]:
# ── G: Hybrid retrieval — BM25 + vector fused with RRF ────────────────────
# Hybrid uses a custom pipeline subclass because it needs the full chunk list for BM25.

class HybridRAGPipeline(RAGPipeline):
    """RAGPipeline with hybrid (BM25 + vector) retrieval."""
    def __init__(self, all_chunks, **kwargs):
        super().__init__(**kwargs)
        self.all_chunks = all_chunks

    def _retrieve(self, question):
        return _hybrid_retrieve(
            collection=self.collection,
            all_chunks=self.all_chunks,
            question=question,
            top_k=self.top_k,
            model_name=self.embed_model_name,
        )


collection_hybrid = get_collection("exp_r2_hybrid", persist_path="../chroma_db_experiments")
index_chunks(collection_hybrid, best_chunks, model_name=DEFAULT_MODEL)

hybrid_pipeline = HybridRAGPipeline(
    all_chunks=best_chunks,
    collection=collection_hybrid,
    llm_client=llm_client,
    llm_model=FAST_MODEL,
    top_k=5,
)

hybrid_results = []
for qa in tqdm(qa_data["questions"], desc="[Round2-Hybrid]"):
    r = hybrid_pipeline.ask(qa["question"])
    r["ground_truth"] = qa["ground_truth"]
    hybrid_results.append(r)

from solutions.src.evaluator import run_evaluation
df_hybrid = run_evaluation(hybrid_results, llm_client=llm_client, judge_model=FAST_MODEL)
scores_hybrid = {c: round(float(df_hybrid[c].mean()), 4)
                 for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
                 if c in df_hybrid.columns}

log.add(
    name="Round2-Hybrid",
    config={"strategy": BEST_CHUNK_STRATEGY, "top_k": 5, "retrieval": "hybrid-BM25+vector-RRF"},
    scores=scores_hybrid,
    notes="BM25 + vector fused with Reciprocal Rank Fusion — handles INAMI/RIZIV exact terms",
)
print(scores_hybrid)

In [ ]:
# Round 2 summary
log.summary()
log.delta_table(baseline_name="Baseline")
log.plot()

# Pick winner based on context_precision (retrieval relevance)
df_log = log.to_dataframe()
r2_entries = df_log[df_log["name"].str.startswith("Round2")]
best_r2 = r2_entries.loc[r2_entries["context_precision"].idxmax(), "name"]
print(f"\nRound 2 winner (best context_precision): {best_r2}")

# Set config for Round 3 — update based on your results
BEST_RETRIEVAL = "hybrid"  # or "vector"
BEST_TOP_K = 5
print(f"Using for Round 3: {BEST_RETRIEVAL}, top_k={BEST_TOP_K}")

---
## Round 3 — Prompt Templates

**Hypothesis:** Explicit grounding instructions and multilingual directives improve faithfulness
without reducing answer relevancy.

**Variable:** system prompt  
**Fixed:** best chunking from Round 1, best retrieval from Round 2

| Config | Prompt style |
|--------|--------------|
| H | Baseline (minimal) |
| I | Strict grounding — explicit "say so if insufficient" |
| J | Multilingual — respond in the question's language |
| K | Structured output — Answer / Details / Source sections |

In [ ]:
PROMPT_BASELINE = """You are a helpful insurance assistant. Answer the question using the provided context."""

PROMPT_STRICT = """You are a DKV Belgium insurance assistant.
Answer ONLY using the numbered context chunks provided. Cite chunk numbers.
If the context is insufficient, say exactly: "I cannot answer based on the available context."
Never fabricate policy details, amounts, or conditions."""

PROMPT_MULTILINGUAL = """You are a DKV Belgium insurance assistant.
Answer ONLY using the numbered context chunks provided. Cite chunk numbers.
IMPORTANT: Always respond in the same language as the question.
If the question is in French, answer in French. If in Dutch, answer in Dutch. If in English, answer in English.
If the context is insufficient, say so in the question's language.
Never fabricate policy details, amounts, or conditions."""

PROMPT_STRUCTURED = """You are a DKV Belgium insurance assistant.
Answer ONLY using the numbered context chunks provided.
Format your answer as:
  **Answer:** [direct answer in 1-2 sentences, in the question's language]
  **Details:** [supporting detail from context, bullet points if applicable]
  **Source:** [chunk numbers used, e.g. [1][3]]
If the context is insufficient, state this clearly in the question's language.
Never fabricate policy details, amounts, or conditions."""

prompts = {
    "Round3-Baseline-Prompt": PROMPT_BASELINE,
    "Round3-Strict-Prompt": PROMPT_STRICT,
    "Round3-Multilingual-Prompt": PROMPT_MULTILINGUAL,
    "Round3-Structured-Prompt": PROMPT_STRUCTURED,
}
print(f"Testing {len(prompts)} prompt variants")

In [ ]:
# All Round 3 variants use the best chunking + best collection from Round 2
best_collection = get_collection("exp_r3_best", persist_path="../chroma_db_experiments")
index_chunks(best_collection, best_chunks, model_name=DEFAULT_MODEL)

for exp_name, system_prompt in prompts.items():
    rag = RAGPipeline(
        collection=best_collection,
        llm_client=llm_client,
        llm_model=FAST_MODEL,
        top_k=BEST_TOP_K,
        system_prompt=system_prompt,
    )
    results = []
    for qa in tqdm(qa_data["questions"], desc=f"[{exp_name}]"):
        r = rag.ask(qa["question"])
        r["ground_truth"] = qa["ground_truth"]
        results.append(r)

    df = run_evaluation(results, llm_client=llm_client, judge_model=FAST_MODEL)
    scores = {c: round(float(df[c].mean()), 4)
              for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
              if c in df.columns}

    log.add(
        name=exp_name,
        config={"strategy": BEST_CHUNK_STRATEGY, "top_k": BEST_TOP_K, "retrieval": BEST_RETRIEVAL, "prompt": exp_name},
        scores=scores,
        notes=f"Prompt variant — {exp_name.replace('Round3-', '').replace('-Prompt', '')}",
    )
    print(f"{exp_name}: {scores}")

In [ ]:
# Complete experiment log
print("=" * 70)
print("COMPLETE EXPERIMENT LOG")
print("=" * 70)
log.summary()

print("\n" + "=" * 70)
print("DELTA VS BASELINE")
print("=" * 70)
log.delta_table(baseline_name="Baseline")

log.plot()

---
## Round 4 — LLM Model Comparison

**Hypothesis:** Sonnet produces more faithful, relevant answers than Haiku  
but at higher latency and cost. Running both in parallel quantifies the trade-off.

**Variable:** LLM model  
**Fixed:** best chunking, best retrieval, best prompt from Rounds 1–3


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

SMART_MODEL = os.getenv("NETLIGHT_MODEL_SMART", "claude-sonnet-4-5")
MODELS_TO_COMPARE = {
    "haiku":  FAST_MODEL,
    "sonnet": SMART_MODEL,
}

# Re-use best collection from Round 3
r4_collection = get_collection("exp_r3_best", persist_path="../chroma_db_experiments")

def run_pipeline_for_model(label: str, model_id: str) -> dict:
    pipeline = RAGPipeline(
        collection=r4_collection,
        llm_client=llm_client,
        llm_model=model_id,
        top_k=BEST_TOP_K,
    )
    results, latencies = [], []
    for qa in qa_data["questions"]:
        t0 = time.perf_counter()
        r = pipeline.ask(qa["question"])
        latencies.append(time.perf_counter() - t0)
        r["ground_truth"] = qa["ground_truth"]
        results.append(r)
    return {"label": label, "model": model_id, "results": results, "latencies": latencies}


print("Running pipelines in parallel...")
t_wall = time.perf_counter()

model_outputs = {}
with ThreadPoolExecutor(max_workers=len(MODELS_TO_COMPARE)) as executor:
    futures = {
        executor.submit(run_pipeline_for_model, label, model_id): label
        for label, model_id in MODELS_TO_COMPARE.items()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Models"):
        label = futures[future]
        model_outputs[label] = future.result()
        print(f"  {label} done")

wall_time = time.perf_counter() - t_wall
seq_time = sum(sum(v["latencies"]) for v in model_outputs.values())
print(f"\nDone in {wall_time:.1f}s (sequential would be ~{seq_time:.1f}s)")


In [ ]:
# Evaluate both sets in parallel
print("Evaluating in parallel...")
eval_results = {}
with ThreadPoolExecutor(max_workers=len(MODELS_TO_COMPARE)) as executor:
    futures = {
        executor.submit(
            run_evaluation,
            model_outputs[label]["results"],
            llm_client,
            FAST_MODEL,
        ): label
        for label in MODELS_TO_COMPARE
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Evaluating"):
        label = futures[future]
        eval_results[label] = future.result()

# Log results
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
for label in MODELS_TO_COMPARE:
    df = eval_results[label]
    scores = {c: round(float(df[c].mean()), 4) for c in metric_cols if c in df.columns}
    mean_lat = round(sum(model_outputs[label]["latencies"]) / len(model_outputs[label]["latencies"]), 2)
    log.add(
        name=f"Round4-{label}",
        config={"strategy": BEST_CHUNK_STRATEGY, "top_k": BEST_TOP_K,
                "retrieval": BEST_RETRIEVAL, "llm_model": MODELS_TO_COMPARE[label]},
        scores=scores,
        notes=f"Model: {MODELS_TO_COMPARE[label]} | mean latency: {mean_lat}s",
    )
    print(f"  {label}: {scores}  (mean latency: {mean_lat}s)")


In [ ]:
# Side-by-side comparison: quality metrics + latency vs quality
labels = list(MODELS_TO_COMPARE.keys())
colors = {"haiku": "#3498db", "sonnet": "#e67e22"}

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, :2])
x = np.arange(len(metric_cols))
width = 0.35
for i, label in enumerate(labels):
    scores = eval_results[label][metric_cols].mean().values
    bars = ax1.bar(x + (i - 0.5) * width, scores, width,
                   label=f"{label} ({MODELS_TO_COMPARE[label]})",
                   color=colors.get(label, f"C{i}"), alpha=0.85)
    for bar, score in zip(bars, scores):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{score:.2f}", ha="center", fontsize=8)
ax1.set_xticks(x)
ax1.set_xticklabels(metric_cols, rotation=10)
ax1.set_ylim(0, 1.15)
ax1.set_ylabel("Score")
ax1.set_title("Quality Metrics by Model")
ax1.axhline(0.75, color="gray", linestyle="--", alpha=0.4, label="Good threshold")
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(gs[0, 2])
for label in labels:
    mean_lat = np.mean(model_outputs[label]["latencies"])
    mean_faith = eval_results[label]["faithfulness"].mean()
    ax2.scatter(mean_lat, mean_faith, s=180, color=colors.get(label, "gray"), zorder=5)
    ax2.annotate(label, (mean_lat, mean_faith), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax2.set_xlabel("Mean latency per call (s)")
ax2.set_ylabel("Faithfulness")
ax2.set_title("Latency vs Quality")

plt.suptitle("Round 4 — LLM Model Comparison", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

log.summary()
log.delta_table(baseline_name="Baseline")


---
## Multilingual Spot-Check

Run the best pipeline on FR / NL / EN questions and manually verify language fidelity and factual accuracy.
RAGAS scores don't catch a response in the wrong language — human review does.

In [ ]:
# Find the best prompt from Round 3 by faithfulness
df_log = log.to_dataframe()
r3_entries = df_log[df_log["name"].str.startswith("Round3")]
best_prompt_name = r3_entries.loc[r3_entries["faithfulness"].idxmax(), "name"] if len(r3_entries) else "Round3-Multilingual-Prompt"
best_prompt = prompts.get(best_prompt_name, PROMPT_MULTILINGUAL)
print(f"Best prompt by faithfulness: {best_prompt_name}")

best_rag = RAGPipeline(
    collection=best_collection,
    llm_client=llm_client,
    llm_model=FAST_MODEL,
    top_k=BEST_TOP_K,
    system_prompt=best_prompt,
)

spot_check_questions = [
    ("FR", "Quelle est la franchise annuelle pour l'hospitalisation?"),
    ("NL", "Wat is het maximale dagbedrag voor kinesitherapie bij een geconventioneerde kinesitherapeut?"),
    ("EN", "What documents are required to submit a hospitalisation reimbursement claim?"),
    ("FR", "Les soins dentaires sont-ils remboursés après un accident de la route?"),
    ("NL", "Worden de premies terugbetaald als ik het contract vroeg opzeg?"),
]

print("\n" + "=" * 70)
for lang, question in spot_check_questions:
    result = best_rag.ask(question)
    print(f"\n[{lang}] {question}")
    print(f"  → {result['answer'][:300]}")
    print(f"  Sources: {sorted(set(result['sources']))}")

print("\n\nManual review checklist:")
print("  [ ] FR question answered in French")
print("  [ ] NL question answered in Dutch")
print("  [ ] EN question answered in English")
print("  [ ] No fabricated amounts or policy details")
print("  [ ] Correct source documents cited")

---
## Improvement Backlog

Translate experiment results into a prioritised backlog for the production project.

In [ ]:
df_log = log.to_dataframe()
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("Best configuration per metric (measured evidence):")
print("-" * 55)
for m in metrics:
    if m in df_log.columns:
        best_idx = df_log[m].idxmax()
        row = df_log.loc[best_idx]
        baseline_val = df_log[df_log["name"] == "Baseline"][m].values[0] if "Baseline" in df_log["name"].values else 0
        delta = row[m] - baseline_val
        sign = "+" if delta >= 0 else ""
        print(f"  {m:<28}: {row['name']} ({row[m]:.3f}, {sign}{delta:.3f} vs baseline)")

print("""

IMPROVEMENT BACKLOG
===================

Priority 1 — High impact, measured in today's experiments:
  [ ] <Fill in: best chunking config and delta>
  [ ] <Fill in: best retrieval config and delta>
  [ ] <Fill in: best prompt and delta>

Priority 2 — Moderate impact, worth a second experiment sprint:
  [ ] Cross-encoder re-ranking: retrieve 20, re-rank to top 5 (AdvancedRAGPipeline)
  [ ] Minimum similarity threshold: filter out low-confidence retrievals
  [ ] Increase top_k to 10 + re-ranking to recover recall without precision loss

Priority 3 — Untested, next sprint:
  [ ] Query expansion for FR/NL/EN synonyms (INAMI ↔ RIZIV ↔ NIHDI)
  [ ] HyDE (hypothetical document embedding) for complex policy questions
  [ ] Fine-tune embedding model on DKV Belgium terminology
  [ ] Native-speaker review of FR/NL answer quality
  [ ] Production deployment: FastAPI endpoint + Docker compose update
""")

# Export log for sharing
csv_path = "../data/experiment_log_export.csv"
df_log.to_csv(csv_path, index=False)
print(f"Experiment log exported → {csv_path}")
print(f"\nEnd of workshop. {len(log)} experiments logged.")